# Librerias 

In [16]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Perceptron as SKPerceptron
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# Datos de las Compuertas 

In [17]:
def gate_data(name: str):
    X = np.array([
        [0,0],
        [0,1],
        [1,0],
        [1,1]
    ], dtype=float)

    if name.upper() == "AND":
        y = np.array([0,0,0,1], dtype=float)
    elif name.upper() == "OR":
        y = np.array([0,1,1,1], dtype=float)
    elif name.upper() == "XOR":
        y = np.array([0,1,1,0], dtype=float)
    else:
        raise ValueError("name debe ser AND, OR o XOR")
    return X, y

# Regresion Lineal

In [18]:
class LinearRegressionNE:
    """Regresión lineal por solución analítica"""
    def __init__(self, fit_intercept=True, ridge=0.0):
        self.fit_intercept = fit_intercept
        self.ridge = ridge
        self.w = None

    def _add_bias(self, X):
        if not self.fit_intercept:
            return X
        return np.c_[np.ones((X.shape[0], 1)), X]

    def fit(self, X, y):
        Xb = self._add_bias(X)
        y = y.reshape(-1, 1)

        # (X^T X + λI)^(-1) X^T y
        XtX = Xb.T @ Xb
        if self.ridge > 0:
            I = np.eye(XtX.shape[0])
            if self.fit_intercept:
                I[0,0] = 0.0  
            XtX = XtX + self.ridge * I

        self.w = np.linalg.pinv(XtX) @ (Xb.T @ y)
        return self

    def predict(self, X):
        Xb = self._add_bias(X)
        return (Xb @ self.w).ravel()

    def predict_class(self, X, threshold=0.5):
        return (self.predict(X) >= threshold).astype(int)

# Regresion Logistica 

In [19]:
class LogisticRegressionGD:
    """Regresión logística binaria con gradiente descendente"""
    def __init__(self, lr=0.1, epochs=2000, fit_intercept=True, l2=0.0, seed=0):
        self.lr = lr
        self.epochs = epochs
        self.fit_intercept = fit_intercept
        self.l2 = l2
        self.seed = seed
        self.w = None

    def _add_bias(self, X):
        if not self.fit_intercept:
            return X
        return np.c_[np.ones((X.shape[0], 1)), X]

    @staticmethod
    def _sigmoid(z):
        z = np.clip(z, -50, 50)
        return 1.0 / (1.0 + np.exp(-z))

    def fit(self, X, y):
        Xb = self._add_bias(X)
        y = y.reshape(-1, 1)

        rng = np.random.default_rng(self.seed)
        self.w = rng.normal(0, 0.1, size=(Xb.shape[1], 1))

        n = Xb.shape[0]
        for _ in range(self.epochs):
            p = self._sigmoid(Xb @ self.w)          
            grad = (Xb.T @ (p - y)) / n             

            # L2 
            if self.l2 > 0:
                reg = self.l2 * self.w
                if self.fit_intercept:
                    reg[0,0] = 0.0
                grad = grad + reg

            self.w -= self.lr * grad

        return self

    def predict_proba(self, X):
        Xb = self._add_bias(X)
        return self._sigmoid(Xb @ self.w).ravel()

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

# Perceptron 

In [20]:
class Perceptron:
    """Perceptrón binario clásico """
    def __init__(self, lr=0.1, epochs=50, fit_intercept=True, shuffle=True, seed=0):
        self.lr = lr
        self.epochs = epochs
        self.fit_intercept = fit_intercept
        self.shuffle = shuffle
        self.seed = seed
        self.w = None

    def _add_bias(self, X):
        if not self.fit_intercept:
            return X
        return np.c_[np.ones((X.shape[0], 1)), X]

    def fit(self, X, y):
        Xb = self._add_bias(X)
        y = y.astype(int).ravel()

        rng = np.random.default_rng(self.seed)
        self.w = np.zeros(Xb.shape[1])

        idx = np.arange(Xb.shape[0])
        for _ in range(self.epochs):
            if self.shuffle:
                rng.shuffle(idx)
            for i in idx:
                xi = Xb[i]
                yi = y[i]
                yhat = 1 if (xi @ self.w) >= 0 else 0
                self.w += self.lr * (yi - yhat) * xi

        return self

    def predict(self, X):
        Xb = self._add_bias(X)
        return ((Xb @ self.w) >= 0).astype(int)

# Perceptron Multicapa

In [21]:
class ELMClassifier:
    """
    Multicapa con solución analítica para la salida:
    - W,b de capa oculta aleatorios fijos
    - beta por pseudo-inversa 
    """
    def __init__(self, hidden_units=10, activation="tanh", ridge=1e-6, seed=0):
        self.hidden_units = hidden_units
        self.activation = activation
        self.ridge = ridge
        self.seed = seed
        self.W = None
        self.b = None
        self.beta = None

    def _act(self, Z):
        if self.activation == "tanh":
            return np.tanh(Z)
        if self.activation == "sigmoid":
            Z = np.clip(Z, -50, 50)
            return 1/(1+np.exp(-Z))
        if self.activation == "relu":
            return np.maximum(0, Z)
        raise ValueError("activation debe ser tanh, sigmoid o relu")

    def fit(self, X, y):
        X = X.astype(float)
        y = y.reshape(-1, 1).astype(float)

        rng = np.random.default_rng(self.seed)
        d = X.shape[1]
        self.W = rng.normal(0, 1, size=(d, self.hidden_units))
        self.b = rng.normal(0, 1, size=(1, self.hidden_units))

        H = self._act(X @ self.W + self.b)  

        # Ridge con beta = (H^T H + λI)^-1 H^T y
        HtH = H.T @ H
        I = np.eye(HtH.shape[0])
        self.beta = np.linalg.pinv(HtH + self.ridge * I) @ (H.T @ y)

        return self

    def predict_proba(self, X):
        H = self._act(X @ self.W + self.b)
        z = H @ self.beta
        z = np.clip(z, -50, 50)
        return (1/(1+np.exp(-z))).ravel()

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

# Evaluacion 

In [22]:
def accuracy(y_true, y_pred):
    y_true = np.array(y_true).astype(int).ravel()
    y_pred = np.array(y_pred).astype(int).ravel()
    return (y_true == y_pred).mean()

def run_numpy_models(gate="AND"):
    X, y = gate_data(gate)

    models = {
        "Regresion Lineal": LinearRegressionNE().fit(X, y),
        "Regresion Logistica": LogisticRegressionGD(lr=0.5, epochs=3000).fit(X, y),
        "Perceptron": Perceptron(lr=0.2, epochs=50).fit(X, y),
        "Perceptron Multicapa": ELMClassifier(hidden_units=8, activation="tanh").fit(X, y)
    }

    out = {}
    for name, m in models.items():
        if name.startswith("LinearReg"):
            pred = m.predict_class(X)
        else:
            pred = m.predict(X)
        out[name] = accuracy(y, pred)
    return out

for g in ["AND", "OR", "XOR"]:
    print(g, run_numpy_models(g))

AND {'Regresion Lineal': np.float64(0.75), 'Regresion Logistica': np.float64(1.0), 'Perceptron': np.float64(1.0), 'Perceptron Multicapa': np.float64(0.25)}
OR {'Regresion Lineal': np.float64(0.5), 'Regresion Logistica': np.float64(1.0), 'Perceptron': np.float64(1.0), 'Perceptron Multicapa': np.float64(0.75)}
XOR {'Regresion Lineal': np.float64(0.5), 'Regresion Logistica': np.float64(0.5), 'Perceptron': np.float64(0.5), 'Perceptron Multicapa': np.float64(0.5)}


# Clasificacion con sklearn

In [23]:

def run_sklearn(gate="AND"):
    X, y = gate_data(gate)
    y = y.astype(int)

    models = {
        "Regresion Logistica": LogisticRegression(C=1e6, solver="lbfgs"),
        "Perceptron": SKPerceptron(max_iter=1000, tol=1e-4, random_state=0),
        "Multicapa con 1 capa oculta": MLPClassifier(hidden_layer_sizes=(4,), activation="tanh",
                                      max_iter=5000, random_state=0),
    }

    results = {}
    for name, clf in models.items():
        clf.fit(X, y)
        pred = clf.predict(X)
        results[name] = accuracy_score(y, pred)

    return results

for g in ["AND","OR","XOR"]:
    print(g, run_sklearn(g))

AND {'Regresion Logistica': 1.0, 'Perceptron': 1.0, 'Multicapa con 1 capa oculta': 1.0}
OR {'Regresion Logistica': 1.0, 'Perceptron': 1.0, 'Multicapa con 1 capa oculta': 1.0}
XOR {'Regresion Logistica': 0.5, 'Perceptron': 0.5, 'Multicapa con 1 capa oculta': 1.0}


# Análisis de Resultados -- Clasificación de Compuertas Lógicas


## 1. Análisis por Compuerta

### AND

Es linealmente separable.

-   En NumPy:
    -   Regresión Logística y Perceptrón alcanzaron 1.0.
    -   Regresión Lineal logró 0.75 (al usar umbral).
    -   El Perceptrón Multicapa obtuvo 0.25, lo cual sugiere problemas
        de inicialización o configuración.
-   En Scikit-Learn:
    -   Todos los modelos alcanzaron 1.0.

Podemos ver que AND es separable con modelos lineales.

------------------------------------------------------------------------

### OR

También es linealmente separable.

-   En NumPy:
    -   Logística y Perceptrón alcanzaron 1.0.
    -   Regresión Lineal logró 0.5.
    -   Multicapa obtuvo 0.75.
-   En Scikit-Learn:
    -   Todos alcanzaron 1.0.

OR también es un problema lineal simple.

------------------------------------------------------------------------

### XOR

No es linealmente separable.

-   En NumPy:
    -   Todos los modelos obtuvieron 0.5.
    -   Esto indica que el modelo multicapa analítico no logró generar
        suficiente no linealidad o neuronas ocultas efectivas.
-   En Scikit-Learn:
    -   Logística y Perceptrón  obtuvieron 0.5.
    -   El Muticapa con una capa oculta logró 1.0.

XOR necesita la no linealidad real en la frontera de decisión.

------------------------------------------------------------------------

## 2. Comparación NumPy vs Scikit-Learn

### Implementación desde Cero

Ventajas: - Mayor comprensión matemática. - Control total del
algoritmo. - Bueno para el aprendizaje teórico.

Desventajas: - Sensible a inicialización. - Requiere ajuste manual de
hiperparámetros. - Puede fallar 

### Implementación con Librerías

Ventajas: - Más uso de los recursos. - Mejor optimización. - Mejor convergencia.

Desventajas:  - Depende de sus hiperparámetros 

------------------------------------------------------------------------

## 3. Conclusiones Generales

1.  AND y OR son problemas linealmente separables y funcionan
    perfectamente con modelos lineales.
2.  XOR requiere modelos no lineales.
3.  La implementación manual puede no capturar de la mejor manera la
    complejidad si no se configuran las neuronas ocultas.
4.  Scikit-Learn ofrece mejor desempeño en problemas no lineales peus tiene una mejor optimizacion.

------------------------------------------------------------------------

## 4. Reflexión Final

Podemos ver cómo la separabilidad lineal es importante para el desempeño de los modelos. Los modelos lineales funcionan cuando existe un hiperplano que pueda separar. Cuando el problema no es lineal, necesitamos una transformación del espacio de características que se usa por lo general con las capas ocultas o kernels(como en SVM).